In [1]:
from pycircstat2 import Circular

# Imports

import helpers.helper_functions as hf
import mne
import os.path as op
from mne.channels import combine_channels
import pandas as pd
from mne.beamformer import make_lcmv, apply_lcmv_epochs
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
from scipy.signal import hilbert
import helpers.test_circ_plot as circ_plot
import gc
import helpers.stc_helper as stc_helper
import time
from scipy.stats import norm

ss = hf.settings_dict()
from mne.stats import fdr_correction
from pycircstat2.hypothesis import wheeler_watson_test, rayleigh_test

In [2]:
stim_tmin = 0.7
stim_tmax = 3.7
base_tmin = -0.5
base_tmax = 0.0

In [3]:
for subject_index in ss['subject_idx_list']:

    # loop over each event type
    for event_id in ss['event_id_list']:

        event_name = str(event_id)
        duty_cycle = ss['event_name_list'][event_id - 1]
        subjects_dir = ss['fs_subjects_dir']
        subject = ss['subject_id_list'][subject_index]
        print("loading dataset for subject: ", subject)

        save_dir = Path(ss['hilbert_ref_dir']) / subject / event_name
        save_dir.mkdir(parents=True, exist_ok=True)

        # read the retina csv
        reference_file = Path(ss['hilbert_dir']) / subject / event_name / f"{subject}-trial-{0}-reference.csv"

        df = pd.read_csv(reference_file)

        # get voxel number
        ref_voxel = int(df.axes[1][2].split("_")[0])

        stim_stc_data = []
        base_stc_data = []
        for i in range(ss['n_trials']):
            hilbert_stc_file = Path(ss['hilbert_ref_dir']) / subject / event_name / f"{subject}-trial{i}-ret-ref-wrp-vol.stc"
            stc = mne.read_source_estimate(hilbert_stc_file)
            stim_stc = stc.copy().crop(tmin=stim_tmin, tmax=stim_tmax)
            base_stc = stc.copy().crop(tmin=base_tmin, tmax=base_tmax)
            stim_stc_data.append(stim_stc.data)
            base_stc_data.append(base_stc.data)

        stim_stc_data_stacked = np.stack(stim_stc_data)
        base_stc_data_stacked = np.stack(base_stc_data)

        # stc.data shape: (n_voxels, n_times) — phase differences in radians [-pi, pi]
        n_voxels, n_times = stc.data.shape

        z_stats = np.zeros(n_voxels)
        w_stats = np.zeros(n_voxels)
        p_val = np.zeros(n_voxels)

        for voxel_idx in range(n_voxels):
            avg_stim = np.angle(np.exp(1j * stim_stc_data_stacked[:, voxel_idx]).mean(axis=1))
            avg_stim_ref_vox = np.angle(np.exp(1j * stim_stc_data_stacked[:, ref_voxel]).mean(axis=1))

            avg_base = np.angle(np.exp(1j * base_stc_data_stacked[:, voxel_idx]).mean(axis=1))
            avg_base_ref_vox = np.angle(np.exp(1j * base_stc_data_stacked[:, ref_voxel]).mean(axis=1))

            # Rayleigh test to get z-scores
            result_stim = rayleigh_test(alpha=avg_stim)
            result_base = rayleigh_test(alpha=avg_base)

            # Wheeler Watson w
            ww = wheeler_watson_test([avg_stim,avg_stim_ref_vox]).W

            w_stats[voxel_idx] = ww

            z_stats[voxel_idx] = (result_stim.z - result_base.z)
            p_val[voxel_idx] = result_stim.pval

        # normalize w
        w_stats_normalized = w_stats / np.max(w_stats)
        # explicitly set max mean amplitude voxel to 0
        w_stats_normalized[ref_voxel] = 1

        z_w_stats = z_stats * w_stats_normalized

        H0, p_val_corr = fdr_correction(p_val, 0.05)

        z_stc       = stc.copy().crop(0.0, 0.0 + 1/ss['fs'])
        z_stc.data  = z_w_stats[:, np.newaxis]   # (n_voxels, 1)

        p_stc       = stc.copy().crop(0.0, 0.0 + 1/ss['fs'])
        p_stc.data  = p_val_corr[:, np.newaxis]   # (n_voxels, 1)

        z_stc.save(save_dir / f"{subject}-event-{event_name}-z-vol.stc" , overwrite=True)
        p_stc.save(save_dir / f"{subject}-event-{event_name}-p-vol.stc" , overwrite=True)

        print(f"z min  : {z_w_stats.min():.4f}")
        print(f"z max  : {z_w_stats.max():.4f}")
        print(f"z mean : {z_w_stats.mean():.4f}")
        print(f"Voxels > 0   : {(z_w_stats > 0).sum()} / {n_voxels}")

        print(f"p min  : {p_val_corr.min():.4f}")
        print(f"p max  : {p_val_corr.max():.4f}")
        print(f"p mean : {p_val_corr.mean():.4f}")
        print(f"Significtant voxels   : {(p_val_corr < 0.05).sum()} / {n_voxels}")


loading dataset for subject:  0005_3SJ
Overwriting existing file.
Writing STC to disk...
Overwriting existing file.
[done]
Overwriting existing file.
Writing STC to disk...
Overwriting existing file.
[done]
z min  : -3.1817
z max  : 9.1330
z mean : 0.8164
Voxels > 0   : 911 / 1440
p min  : 0.0009
p max  : 0.9998
p mean : 0.5047
Significtant voxels   : 244 / 1440
loading dataset for subject:  0005_3SJ
Overwriting existing file.
Writing STC to disk...
Overwriting existing file.
[done]
Overwriting existing file.
Writing STC to disk...
Overwriting existing file.
[done]
z min  : -3.8188
z max  : 9.5845
z mean : 0.7281
Voxels > 0   : 970 / 1440
p min  : 0.0006
p max  : 0.9961
p mean : 0.4611
Significtant voxels   : 201 / 1440
loading dataset for subject:  0005_3SJ
Overwriting existing file.
Writing STC to disk...
Overwriting existing file.
[done]
Overwriting existing file.
Writing STC to disk...
Overwriting existing file.
[done]
z min  : -3.9985
z max  : 8.9766
z mean : 0.6939
Voxels > 0   :